# SmartParking — Huấn luyện Pipeline Hướng B trên Google Colab

**Kiến trúc tự xây thay thế OCR ngoài:**
1. **YOLOv8n (1 class: `char`)**: Cắt ảnh biển số thành từng ảnh ký tự nhỏ (`char_detect.onnx`).
2. **HOG + SVM (so sánh với Random Forest & KNN)**: Phân loại từng ảnh ký tự `32×32` thành 31–36 lớp (`char_svm_model.joblib`).
3. **Sắp xếp 1–2 dòng + Constrained Decoding**: Ghép thành chuỗi biển số Việt Nam hoàn chỉnh.

> **Lưu ý:** Bật GPU trên Colab trước khi chạy: `Runtime → Change runtime type → T4 GPU`.

In [ ]:
# 1. Cài đặt thư viện cần thiết
!pip -q install ultralytics roboflow onnx onnxruntime scikit-learn opencv-python-headless joblib seaborn

## Bước 1: Tải Dataset biển số VN có gán nhãn từng ký tự từ Roboflow

- Vào https://universe.roboflow.com tìm từ khóa: `vietnam license plate character recognition` (chọn bộ có gán nhãn từng ký tự `0-9`, `A-Z`, định dạng **YOLOv8**).
- Dán đoạn code tải của Roboflow vào ô bên dưới (hoặc upload file `.zip` lên Google Drive rồi giải nén vào `/content/raw_dataset`).

In [ ]:
from pathlib import Path

# --- CÁCH 1: Tải trực tiếp từ Roboflow (điền thông tin project của bạn) ---
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
# project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
# dataset = project.version(1).download("yolov8", location="/content/raw_dataset")

# --- CÁCH 2: Giải nén file zip đã tải từ Roboflow lên Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')
# !unzip -q '/content/drive/MyDrive/vn_plate_chars_yolov8.zip' -d /content/raw_dataset

RAW_DIR = Path("/content/raw_dataset")
print("Kiểm tra thư mục dataset:", list(RAW_DIR.glob("*")) if RAW_DIR.exists() else "Chưa có dữ liệu")

## Bước 2: Chuyển nhãn sang 1 class `char` cho YOLOv8n + Cắt ảnh ký tự cho HOG + SVM

- Giữ nguyên chia tập `train / valid / test` theo **từng ảnh biển số** để tuyệt đối không bị Data Leakage.
- Mở rộng box thêm 6% khi cắt ký tự để không bị mất nét.

In [ ]:
import shutil
import yaml
import cv2

YOLO_1CLASS_DIR = Path("/content/yolo_char_1class")
CHAR_CROPS_DIR = Path("/content/char_dataset")

with open(RAW_DIR / "data.yaml", "r", encoding="utf-8") as f:
    raw_cfg = yaml.safe_load(f)

raw_names = raw_cfg["names"]
if isinstance(raw_names, dict):
    id_to_name = {int(k): str(v).strip().upper() for k, v in raw_names.items()}
else:
    id_to_name = {i: str(v).strip().upper() for i, v in enumerate(raw_names)}

print(f"Số lớp gốc trong dataset: {len(id_to_name)} -> {list(id_to_name.values())[:15]}...")

crop_counts = {"train": 0, "valid": 0, "test": 0}
for split in ("train", "valid", "test"):
    src_img_dir = RAW_DIR / split / "images"
    src_lbl_dir = RAW_DIR / split / "labels"
    if not src_img_dir.is_dir():
        continue

    dst_img_dir = YOLO_1CLASS_DIR / split / "images"
    dst_lbl_dir = YOLO_1CLASS_DIR / split / "labels"
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    for img_path in sorted(src_img_dir.iterdir()):
        if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
            continue
        lbl_path = src_lbl_dir / f"{img_path.stem}.txt"
        if not lbl_path.exists():
            continue

        shutil.copy2(img_path, dst_img_dir / img_path.name)
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]

        single_class_lines = []
        for idx, line in enumerate(lbl_path.read_text(encoding="utf-8").splitlines()):
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls_id = int(float(parts[0]))
            xc, yc, bw, bh = map(float, parts[1:5])
            # Đổi mọi lớp thành class 0 ('char') cho YOLOv8n
            single_class_lines.append(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

            # Cắt ảnh ký tự nhỏ lưu theo lớp để train HOG + SVM
            label = id_to_name.get(cls_id, "")
            if len(label) == 1 and label.isalnum():
                px, py = bw * w * 0.06, bh * h * 0.06
                x0 = max(0, int((xc - bw / 2) * w - px))
                y0 = max(0, int((yc - bh / 2) * h - py))
                x1 = min(w, int((xc + bw / 2) * w + px))
                y1 = min(h, int((yc + bh / 2) * h + py))
                crop = img[y0:y1, x0:x1]
                if crop.size > 0 and crop.shape[0] >= 6 and crop.shape[1] >= 4:
                    out_dir = CHAR_CROPS_DIR / split / label
                    out_dir.mkdir(parents=True, exist_ok=True)
                    cv2.imwrite(str(out_dir / f"{img_path.stem}_char{idx}.jpg"), crop)
                    crop_counts[split] += 1

        (dst_lbl_dir / lbl_path.name).write_text("\n".join(single_class_lines), encoding="utf-8")

yaml_1class = {
    "path": str(YOLO_1CLASS_DIR),
    "train": "train/images",
    "val": "valid/images" if (YOLO_1CLASS_DIR / "valid").exists() else "test/images",
    "names": {0: "char"},
}
if (YOLO_1CLASS_DIR / "test").exists():
    yaml_1class["test"] = "test/images"

with open(YOLO_1CLASS_DIR / "data.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(yaml_1class, f, sort_keys=False)

print("Đã chuẩn bị xong dataset! Số ảnh ký tự đã cắt:", crop_counts)

## Bước 3: Huấn luyện YOLOv8n (1 class `char`) và xuất `char_detect.onnx`

- **Bắt buộc đặt `fliplr=0.0` và `flipud=0.0`** để không lật ngược ký tự biển số.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=str(YOLO_1CLASS_DIR / "data.yaml"),
    epochs=80,
    imgsz=320,
    batch=32,
    fliplr=0.0,
    flipud=0.0,
    degrees=8.0,
    patience=20,
    project="/content/runs_char",
    name="yolov8n_char_1class",
)

best_pt = Path("/content/runs_char/yolov8n_char_1class/weights/best.pt")
best_model = YOLO(str(best_pt))
onnx_path = best_model.export(format="onnx", imgsz=320, simplify=True)
shutil.copy2(onnx_path, "/content/char_detect.onnx")
print("Đã xuất model YOLO cắt ký tự: /content/char_detect.onnx")

## Bước 4: Huấn luyện HOG + SVM (và đối chiếu Random Forest, KNN)

- Trích xuất đặc trưng HOG giống hệt `src/features/char_hog.py` trong repo `SMART_PARKING_ML` (ảnh xám → CLAHE → Letterbox pad vuông → `32×32` → HOG 324 chiều).

In [ ]:
import time
import joblib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def compute_hog_32x32(gray_img):
    img = gray_img.astype(np.float32) / 255.0
    h, w = img.shape[:2]
    gx, gy = np.zeros_like(img), np.zeros_like(img)
    gx[:, 1:-1] = img[:, 2:] - img[:, :-2]
    gy[1:-1, :] = img[2:, :] - img[:-2, :]
    mag = np.hypot(gx, gy)
    ori = np.degrees(np.arctan2(gy, gx)) % 180.0
    cell_hist = np.zeros((h // 8, w // 8, 9), dtype=np.float32)
    for cy in range(h // 8):
        for cx in range(w // 8):
            mc = mag[cy*8:(cy+1)*8, cx*8:(cx+1)*8].ravel()
            ac = ori[cy*8:(cy+1)*8, cx*8:(cx+1)*8].ravel()
            bf = ac / 20.0
            b0 = np.floor(bf).astype(np.int32) % 9
            b1 = (b0 + 1) % 9
            w1 = bf - np.floor(bf)
            np.add.at(cell_hist[cy, cx], b0, mc * (1.0 - w1))
            np.add.at(cell_hist[cy, cx], b1, mc * w1)
    blocks = []
    for by in range(h // 8 - 1):
        for bx in range(w // 8 - 1):
            blk = cell_hist[by:by+2, bx:bx+2, :].ravel()
            blk = np.minimum(blk / np.sqrt(np.sum(blk*blk) + 1e-10), 0.2)
            blocks.append(blk / np.sqrt(np.sum(blk*blk) + 1e-10))
    return np.concatenate(blocks).astype(np.float32)

def extract_hog(crop_bgr):
    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4)).apply(gray)
    h, w = gray.shape[:2]
    border = np.concatenate([gray[0, :], gray[-1, :], gray[:, 0], gray[:, -1]])
    pad_val = int(np.median(border)) if border.size else 255
    side = max(h, w)
    canvas_side = max(side + max(2, int(round(side * 0.12))), 4)
    canvas = np.full((canvas_side, canvas_side), pad_val, dtype=np.uint8)
    y0, x0 = (canvas_side - h) // 2, (canvas_side - w) // 2
    canvas[y0:y0 + h, x0:x0 + w] = gray
    resized = cv2.resize(canvas, (32, 32), interpolation=cv2.INTER_AREA if canvas_side > 32 else cv2.INTER_CUBIC)
    return compute_hog_32x32(resized)

def load_hog_split(split_dir):
    X, y = [], []
    for cls_dir in sorted(split_dir.iterdir()):
        if not cls_dir.is_dir():
            continue
        for p in sorted(cls_dir.glob("*.jpg")):
            img = cv2.imread(str(p))
            if img is not None:
                X.append(extract_hog(img))
                y.append(cls_dir.name)
    return np.asarray(X, dtype=np.float32), np.asarray(y)

X_train, y_train = load_hog_split(CHAR_CROPS_DIR / "train")
test_split = "test" if (CHAR_CROPS_DIR / "test").exists() else "valid"
X_test, y_test = load_hog_split(CHAR_CROPS_DIR / test_split)
print(f"Train: {X_train.shape} | Test ({test_split}): {X_test.shape}")

candidates = {
    "SVM_RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(kernel="rbf", C=10.0, gamma="scale", probability=True, class_weight="balanced", random_state=42)),
    ]),
    "RandomForest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(n_neighbors=5, weights="distance")),
    ]),
}

for name, clf in candidates.items():
    t0 = time.perf_counter()
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro", zero_division=0)
    print(f"{name:14s} | Acc={acc:.4f} | Macro-F1={f1:.4f} | time={time.perf_counter()-t0:.1f}s")

svm_model = candidates["SVM_RBF"]
joblib.dump(svm_model, "/content/char_svm_model.joblib")
print("Đã lưu model phân loại ký tự: /content/char_svm_model.joblib")

## Bước 5: Tải 2 file model về máy để bỏ vào dự án `SMART_PARKING_ML`

- Copy `char_detect.onnx` → `models/vision/char_detect.onnx`
- Copy `char_svm_model.joblib` → `models/classification/char_svm_model.joblib`

In [ ]:
from google.colab import files

files.download("/content/char_detect.onnx")
files.download("/content/char_svm_model.joblib")